In [49]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os

In [50]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

d:\_PROJECTS\_ACTIVE_LOCAL\OHCA_Prediction_PAROS\Mobile-AED-Study\src
d:\_PROJECTS\_ACTIVE_LOCAL\OHCA_Prediction_PAROS\Mobile-AED-Study\datasets


In [51]:
ohca_filepath = Path(BASE_DATASET_PATH / f"3_OHCA_with_geometric_pts.gpkg")
ohca_points = gpd.read_file(ohca_filepath)

c:\ProgramData\anaconda3\envs\snowflakes\Lib\site-packages\geopandas\io\file.py:399: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  as_dt = pd.to_datetime(df[k], errors="ignore")
c:\ProgramData\anaconda3\envs\snowflakes\Lib\site-packages\geopandas\io\file.py:403: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  as_dt = pd.to_datetime(df[k], errors="ignore", utc=True)


In [52]:
ohca_points.head()

,Case #,Site #,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Time call received at dispatch center,Estimated time of arrest,lat,lon,geometry
0,SGSIN0213,2,2010-04-01T00:00:00,470146.0,None,Home Residence,HDB Level 7,23:29:17,23:35:00,1.334604,103.911889,POINT (36744.065 35199.375)
1,SGSIN0218,2,2010-04-01T00:00:00,520926.0,None,Home Residence,HDB Level 2,14:18:54,14:05:00,1.346122,103.940989,POINT (39982.538 36473.112)
2,SGSIN6480,6,2010-04-01T00:00:00,560565.0,None,Healthcare Facility,NKF Dialysis Centre,16:35:03,16:30:00,1.369882,103.858475,POINT (30799.605 39100.122)
3,SGSIN5332,5,2010-04-02T00:00:00,680626.0,None,Home Residence,HDB Level 7,02:28:08,02:00:00,1.398315,103.745837,POINT (18264.472 42244.265)
4,SGSIN0214,2,2010-04-03T00:00:00,468963.0,None,Place of Recreation,East Coast Park NSC Carpark,08:56:21,09:00:00,1.315317,103.961890,POINT (42308.771 33066.957)


In [53]:
subzones_2014_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2014/subzone_boundary_2014.gpkg")
subzones_2019_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2019/subzone_boundary_2019.gpkg")

subzone_2014 = gpd.read_file(subzones_2014_filepath)
subzone_2014.columns = subzone_2014.columns.str.lower()
subzone_2014['subzone_n'] = subzone_2014['subzone_n'].str.strip().str.lower()
subzone_2014['pln_area_n'] = subzone_2014['pln_area_n'].str.strip().str.lower()

subzone_2019 = gpd.read_file(subzones_2019_filepath)
subzone_2019.columns = subzone_2019.columns.str.lower()
subzone_2019['subzone_n'] = subzone_2019['subzone_n'].str.strip().str.lower()
subzone_2019['pln_area_n'] = subzone_2019['pln_area_n'].str.strip().str.lower()

In [54]:
subzone_2014.head()

,objectid,subzone_no,subzone_n,subzone_c,ca_ind,pln_area_n,pln_area_c,region_n,region_c,inc_crc,fmel_upd_d,x_addr,y_addr,shape_leng,shape_area,geometry
0,1,2,people's park,OTSZ02,Y,outram,OT,CENTRAL REGION,CR,B4120D23006C932A,2016-05-11,28831.7807,29419.6457,1822.192718,93140.437642,"MULTIPOLYGON (((29099.021 29640.030, 29116.963..."
1,2,2,bukit merah,BMSZ02,N,bukit merah,BM,CENTRAL REGION,CR,1C51019439A68700,2016-05-11,26360.7990,29384.1429,3074.963235,411722.822583,"MULTIPOLYGON (((26750.092 29216.098, 26751.912..."
2,3,3,chinatown,OTSZ03,Y,outram,OT,CENTRAL REGION,CR,0FF1661344C84AED,2016-05-11,29153.9676,29158.0443,4297.599910,587222.678854,"MULTIPOLYGON (((29161.201 29723.071, 29189.033..."
3,4,4,phillip,DTSZ04,Y,downtown core,DT,CENTRAL REGION,CR,615D4EDDEF809F8E,2016-05-11,29706.7242,29744.9079,871.554888,39437.935270,"MULTIPOLYGON (((29814.107 29616.894, 29806.682..."
4,5,5,raffles place,DTSZ05,Y,downtown core,DT,CENTRAL REGION,CR,72107B11807074F4,2016-05-11,29968.6175,29572.7618,1872.752161,188767.489706,"MULTIPOLYGON (((30137.768 29843.194, 30138.417..."


In [55]:
gdf_ohca = ohca_points.to_crs(epsg=3414)
gdf_subzone_2014 = subzone_2014.to_crs(epsg=3414)
gdf_subzone_2019 = subzone_2019.to_crs(epsg=3414)

In [56]:
gdf_ohca['Date of Incident'] = pd.to_datetime(gdf_ohca['Date of Incident'], format="mixed")

In [57]:
# split the ohca cases 2010 - 2018 and 2019 - 2021 
# they are split because geospatial data changed from 2014 to 2019 masterplan

ohca_2010_2018 = gdf_ohca[gdf_ohca['Date of Incident'].dt.year.between(2010, 2018)]
ohca_2019_2021 = gdf_ohca[gdf_ohca['Date of Incident'].dt.year.between(2019, 2021)]

ohca_2010_2018_joined = gpd.sjoin(ohca_2010_2018, gdf_subzone_2014, how = "left", predicate = "within")
ohca_2019_2021_joined = gpd.sjoin(ohca_2019_2021, gdf_subzone_2019, how = "left", predicate = "within")

print("Columns in 2010-2018 dataframe:", ohca_2010_2018_joined.columns.tolist())
display(ohca_2010_2018_joined[['Case #', 'pln_area_n', 'subzone_n', 'Date of Incident']].head())

Columns in 2010-2018 dataframe: ['Case #', 'Site #', 'Date of Incident', 'Location of incident', 'Location Unknown', 'Location Type', 'Location Type Other', 'Time call received at dispatch center', 'Estimated time of arrest', 'lat', 'lon', 'geometry', 'index_right', 'objectid', 'subzone_no', 'subzone_n', 'subzone_c', 'ca_ind', 'pln_area_n', 'pln_area_c', 'region_n', 'region_c', 'inc_crc', 'fmel_upd_d', 'x_addr', 'y_addr', 'shape_leng', 'shape_area']


,Case #,pln_area_n,subzone_n,Date of Incident
0,SGSIN0213,bedok,kaki bukit,2010-04-01
1,SGSIN0218,tampines,tampines west,2010-04-01
2,SGSIN6480,ang mo kio,cheng san,2010-04-01
3,SGSIN5332,choa chu kang,yew tee,2010-04-02
4,SGSIN0214,bedok,bayshore,2010-04-03


In [58]:
ohca_2010_2018_joined['year'] = ohca_2010_2018_joined['Date of Incident'].dt.year
ohca_2010_2018_joined['month'] = ohca_2010_2018_joined['Date of Incident'].dt.month

ohca_2019_2021_joined['year'] = ohca_2019_2021_joined['Date of Incident'].dt.year
ohca_2019_2021_joined['month'] = ohca_2019_2021_joined['Date of Incident'].dt.month

display(ohca_2010_2018_joined[['Case #', 'pln_area_n', 'subzone_n', 'Date of Incident', 'year', 'month']].head())

,Case #,pln_area_n,subzone_n,Date of Incident,year,month
0,SGSIN0213,bedok,kaki bukit,2010-04-01,2010,4
1,SGSIN0218,tampines,tampines west,2010-04-01,2010,4
2,SGSIN6480,ang mo kio,cheng san,2010-04-01,2010,4
3,SGSIN5332,choa chu kang,yew tee,2010-04-02,2010,4
4,SGSIN0214,bedok,bayshore,2010-04-03,2010,4


In [59]:
# subzones_2014_filepath = Path(BASE_DATASET_PATH / f"ohca_joined_2010_2018.gpkg")
# ohca_2010_2018_joined.to_file(subzones_2014_filepath)
# subzones_2019_filepath = Path(BASE_DATASET_PATH / f"ohca_joined_2019_2021.gpkg")
# ohca_2019_2021_joined.to_file(subzones_2019_filepath)

In [60]:
ohca_2010_2018_joined

sum_ohca_2010_2018_df = ohca_2010_2018_joined.groupby(['pln_area_n', 'subzone_n', 'year', 'month']).size().reset_index(name='incident_count')

sum_ohca_2019_2021_df = ohca_2019_2021_joined.groupby(['pln_area_n', 'subzone_n', 'year', 'month']).size().reset_index(name='incident_count')



In [61]:
save_to_filepath_2010_2018 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_2010_2018.csv")
save_to_filepath_2019_2021 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_2019_2021.csv")
sum_ohca_2010_2018_df.to_csv(save_to_filepath_2010_2018)
sum_ohca_2019_2021_df.to_csv(save_to_filepath_2019_2021)

In [62]:
# create a backbone for logisitc regression, 
# where each subzone, planning area pair has an instance from year 2010 month 4 to year 2021 month 12
# Define the timeframe: April 2010 to December 2021
# combine the 2 ohca dataframes

ohca_all_incidents = pd.concat([sum_ohca_2010_2018_df, sum_ohca_2019_2021_df], ignore_index=True)

master_subzones = ohca_all_incidents[["pln_area_n", "subzone_n"]].drop_duplicates()

years = np.arange(2010, 2022)
months = np.arange(1, 13)

# Create the backbone (Every Subzone x Every Year x Every Month)
backbone = master_subzones.assign(k=1).merge(pd.DataFrame({'year': years, 'k': 1}), on='k')
backbone = backbone.merge(pd.DataFrame({'month': months, 'k': 1}), on='k').drop('k', axis=1)

# Filter to start from April 2010
backbone = backbone[~((backbone['year'] == 2010) & (backbone['month'] < 4))].copy()
backbone

,pln_area_n,subzone_n,year,month
3,ang mo kio,ang mo kio town centre,2010,4
4,ang mo kio,ang mo kio town centre,2010,5
5,ang mo kio,ang mo kio town centre,2010,6
6,ang mo kio,ang mo kio town centre,2010,7
7,ang mo kio,ang mo kio town centre,2010,8
...,...,...,...,...
44491,western water catchment,murai,2021,8
44492,western water catchment,murai,2021,9
44493,western water catchment,murai,2021,10
44494,western water catchment,murai,2021,11


In [63]:
# merge on subzone_n, pln_area_n, year, and month
model_df = pd.merge(
    backbone, 
    ohca_all_incidents, 
    on=['subzone_n', 'pln_area_n', 'year', 'month'], 
    how='left'
)

# Fill missing values with 0 (No incident recorded)
model_df['incident_count'] = model_df['incident_count'].fillna(0)

# Create the binary target: 1 if count > 0, else 0
model_df['ohca_binary'] = (model_df['incident_count'] > 0).astype(int)

# Check the distribution of 0s and 1s
print(model_df['ohca_binary'].value_counts())
display(model_df.head())

ohca_binary
0    28447
1    15122
Name: count, dtype: int64


,pln_area_n,subzone_n,year,month,incident_count,ohca_binary
0,ang mo kio,ang mo kio town centre,2010,4,0.0,0
1,ang mo kio,ang mo kio town centre,2010,5,1.0,1
2,ang mo kio,ang mo kio town centre,2010,6,0.0,0
3,ang mo kio,ang mo kio town centre,2010,7,2.0,1
4,ang mo kio,ang mo kio town centre,2010,8,0.0,0


In [64]:
subzone_2010_2018_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/combined_demographic_geo_2010_2018.csv")
subzone_2010_2018 = pd.read_csv(subzone_2010_2018_filepath)
subzone_2010_2018.columns = subzone_2010_2018.columns.str.lower()
subzone_2010_2018 = subzone_2010_2018.drop(columns=["unnamed: 0"])

subzone_2019_2021_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/combined_demographic_geo_2019_2021.csv")
subzone_2019_2021 = pd.read_csv(subzone_2019_2021_filepath)
subzone_2019_2021.columns = subzone_2019_2021.columns.str.lower()
subzone_2019_2021 = subzone_2019_2021.drop(columns=["unnamed: 0"])
subzone_2019_2021

,subzone_n,pln_area_n,year,total,above_60,above_60_proportion,male_chinese_proportion,female_chinese_proportion,male_malays_proportion,female_malays_proportion,male_indians_proportion,female_indians_proportion,male_others_proportion,female_others_proportion,business_1_encoding,business_2_encoding,business_park_encoding,school_encoding,airport,is_residential
0,admiralty,sembawang,2019,12933,1870,14.459136,34.168406,34.848836,8.520838,9.077554,4.708884,4.693420,1.863450,1.940772,0,0,0,1,0,1
1,admiralty,sembawang,2020,12835,1976,15.395403,34.070900,34.725360,8.546942,9.139073,4.768212,4.721465,1.877678,1.971173,0,0,0,1,0,1
2,admiralty,sembawang,2021,12483,2043,16.366258,34.030281,34.534968,8.675799,9.260594,4.830570,4.742450,1.890571,1.994713,0,0,0,1,0,1
3,airport road,paya lebar,2019,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,0,0,0,0
4,airport road,paya lebar,2020,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063,yuhua west,jurong east,2020,28501,7300,25.613136,34.440897,35.026841,8.550577,8.420757,5.901547,5.466475,1.171889,1.491176,0,0,0,1,0,1
1064,yuhua west,jurong east,2021,27448,7485,27.269746,34.341300,34.884145,8.670941,8.394054,5.971291,5.483095,1.173127,1.511950,0,0,0,1,0,1
1065,yunnan,jurong west,2019,35914,6127,17.060199,33.118004,32.265969,11.324275,11.204544,5.833380,5.078799,0.715598,0.918862,0,0,0,1,0,1
1066,yunnan,jurong west,2020,35538,6526,18.363442,32.967528,32.109291,11.399066,11.275255,5.906354,5.121279,0.720356,0.928583,0,0,0,1,0,1


In [65]:
# 1. Split model_df into the two corresponding time periods
model_pre_2019 = model_df[model_df['year'].between(2010, 2018)].copy()
model_post_2019 = model_df[model_df['year'].between(2019, 2021)].copy()

# 2. Merge the early period (2010-2018)
# We merge on all three keys to ensure the demographics match the specific year
model_pre_2019_enriched = model_pre_2019.merge(
    subzone_2010_2018, 
    on=['subzone_n', 'pln_area_n', 'year'], 
    how='left'
)

# 3. Merge the late period (2019-2021)
model_late_enriched = model_post_2019.merge(
    subzone_2019_2021, 
    on=['subzone_n', 'pln_area_n', 'year'], 
    how='left'
)

# 4. Combine them back into a single model_df
final_model_df = pd.concat([model_pre_2019_enriched, model_late_enriched], ignore_index=True)

# 5. Clean up any duplicate 'Unnamed' or index columns from the merge
cols_to_drop = [c for c in final_model_df.columns if 'Unnamed' in c]
final_model_df = final_model_df.drop(columns=cols_to_drop)

display(final_model_df.head())

# final_model_df = final_model_df.drop(columns=["incident_count", "total", "above_60"])

# change inf values into 0 as they are the result of dividing by 0
final_model_df.replace([np.inf, -np.inf], 0, inplace=True)


,pln_area_n,subzone_n,year,month,incident_count,ohca_binary,total,above_60,above_60_proportion,male_chinese_proportion,...,male_indians_proportion,female_indians_proportion,male_others_proportion,female_others_proportion,business_1_encoding,business_2_encoding,business_park_encoding,school_encoding,airport,is_residential
0,ang mo kio,ang mo kio town centre,2010,4,0.0,0,18757,2599,13.85616,41.739084,...,2.761636,3.01754,1.450125,1.658048,0,0,0,0,0,1
1,ang mo kio,ang mo kio town centre,2010,5,1.0,1,18757,2599,13.85616,41.739084,...,2.761636,3.01754,1.450125,1.658048,0,0,0,0,0,1
2,ang mo kio,ang mo kio town centre,2010,6,0.0,0,18757,2599,13.85616,41.739084,...,2.761636,3.01754,1.450125,1.658048,0,0,0,0,0,1
3,ang mo kio,ang mo kio town centre,2010,7,2.0,1,18757,2599,13.85616,41.739084,...,2.761636,3.01754,1.450125,1.658048,0,0,0,0,0,1
4,ang mo kio,ang mo kio town centre,2010,8,0.0,0,18757,2599,13.85616,41.739084,...,2.761636,3.01754,1.450125,1.658048,0,0,0,0,0,1


In [67]:
save_to_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_binary_complete.csv")
final_model_df.to_csv(save_to_filepath)

After checking the final combined data frame. The male and females chinese proportions in murai, western water catchment is 1000 and 2000. This is due to the small numbers of residents, resulting in this outlier. The ethnicity proportions for murai will be set to 0 for all years

SL: For the 2000, set to missing data, as proportion cannot be 2000. 1000 become 100% for male. There is only 1 person staying there.

In [ ]:
# set all ethnicity proportions for murai to 0
# cols_to_change = [c for c in final_model_df.columns if 'male' in c]
# condition = (final_model_df['subzone_n'] == 'murai') & (final_model_df['pln_area_n'] == 'western water catchment')
# final_model_df.loc[condition, cols_to_change] = 0

# final_model_df.to_csv(save_to_filepath)

In [68]:
cols_to_change = [c for c in final_model_df.columns if 'proportion' in c]
condition = final_model_df[cols_to_change].gt(100).any(axis=1)
filtered_df = final_model_df[condition].copy()



In [69]:
filtered_df['subzone_n'].unique()

array(['murai'], dtype=object)

In [70]:
# Only set cells to NaN where the individual cell value > 100 (not the entire row)
final_model_df.loc[condition, cols_to_change] = final_model_df.loc[condition, cols_to_change].where(
    final_model_df.loc[condition, cols_to_change] <= 100
)

In [71]:
filtered_df = final_model_df[condition].copy()
filtered_df

,pln_area_n,subzone_n,year,month,incident_count,ohca_binary,total,above_60,above_60_proportion,male_chinese_proportion,...,male_indians_proportion,female_indians_proportion,male_others_proportion,female_others_proportion,business_1_encoding,business_2_encoding,business_park_encoding,school_encoding,airport,is_residential
32421,western water catchment,murai,2017,1,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32422,western water catchment,murai,2017,2,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32423,western water catchment,murai,2017,3,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32424,western water catchment,murai,2017,4,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32425,western water catchment,murai,2017,5,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32426,western water catchment,murai,2017,6,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32427,western water catchment,murai,2017,7,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32428,western water catchment,murai,2017,8,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32429,western water catchment,murai,2017,9,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
32430,western water catchment,murai,2017,10,0.0,0,1,0,0.0,100.0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0


In [72]:


final_model_df.to_csv(save_to_filepath)

In [18]:
# subzone_2014_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2014/subzone_classifications_2014.csv")
# subzone_2014 = pd.read_csv(subzone_2014_filepath)
# subzone_2019_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2019/subzone_classifications_2019.csv")
# subzone_2019 = pd.read_csv(subzone_2019_filepath)
# subzone_2008_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2008/subzone_boundary_2008.gpkg")
# subzone_2008 = gpd.read_file(subzone_2008_filepath)


In [19]:
# subzone_2008.columns = subzone_2008.columns.str.lower()
# subzone_2008['subzone_n'] = subzone_2008['subzone_n'].str.strip().str.lower()
# subzone_2008['pln_area_n'] = subzone_2008['pln_area_n'].str.strip().str.lower()
# subzone_2008.head(3)


In [20]:
# mapping_data = [
#     ("jurong east", "boon lay", "yuhua west"),
#     ("jurong west", "boon lay", "boon lay place"),
#     ("sengkang", "buangkok", "anchorvale"),
#     ("bukit panjang", "bukit panjang", "jelebu"),
#     ("choa chu kang", "central", "choa chu kang central"),
#     ("jurong west", "central", "jurong west central"),
#     ("pasir ris", "elias", "pasir ris west"),
#     ("bukit batok", "hong kah", "hong kah north"),
#     ("sengkang", "jalan kayu east", "fernvale"),
#     ("sengkang", "jalan kayu west", "sengkang west"),
#     ("jurong east", "jurong lake", "lakeside"),
#     ("jurong east", "jurong regional centre", "jurong gateway"),
#     ("toa payoh", "kallang", "lorong 8 toa payoh"),
#     ("choa chu kang", "kranji north", "choa chu kang north"),
#     ("toa payoh", "marymount", "toa payoh west"),
#     ("choa chu kang", "pang sua", "choa chu kang north"),
#     ("pasir ris", "pasir ris town", "pasir ris central"),
#     ("toa payoh", "paya lebar", "joo seng"),
#     ("hougang", "rosyth", "kovan"),
#     ("ang mo kio", "sindo", "tagore"),
#     ("punggol", "subzone 2", "punggol town centre"),
#     ("punggol", "subzone 3", "matilda"),
#     ("punggol", "subzone 4", "punggol field"),
#     ("punggol", "subzone 5", "waterway east"),
#     ("sengkang", "sungei serangoon west", "rivervale"),
#     ("hougang", "tai keng", "tai seng"),
#     ("ang mo kio", "town centre", "ang mo kio town centre"),
#     ("sengkang", "trafalgar", "compassvale"),
#     ("jurong east", "yuhua", "yuhua east")
# ]


# map_df = pd.DataFrame(mapping_data, columns=['pln_area_n', 'subzone_n', 'new_subzone'])
# map_df['pln_area_n'] = map_df['pln_area_n'].str.strip().str.lower()
# map_df['subzone_n'] = map_df['subzone_n'].str.strip().str.lower()
# map_df['new_subzone'] = map_df['new_subzone'].str.strip().str.lower()

# subzone_2008['pln_area_n'] = subzone_2008['pln_area_n'].str.strip().str.lower()
# subzone_2008['subzone_n'] = subzone_2008['subzone_n'].str.strip().str.lower()
# subzone_2008 = subzone_2008.merge(
#     map_df, 
#     on=['pln_area_n', 'subzone_n'],
#     how='left'
# )

# # Replace the old subzone with the new one where a match was found
# subzone_2008['subzone_n'] = subzone_2008['new_subzone'].fillna(subzone_2008['subzone_n'])
# subzone_2008.drop(columns=['new_subzone'], inplace=True)
# subzone_2008.head(2)

In [21]:


# # 1. Standardize the strings to ensure a fair comparison
# for df in [subzone_2014, subzone_2019, subzone_2008]:
#     df['subzone_n'] = df['subzone_n'].str.lower().str.strip()
#     df['pln_area_n'] = df['pln_area_n'].str.lower().str.strip()

# # 2. Extract unique sets of (subzone, planning_area) pairs
# pairs_2008 = set(zip(subzone_2008['subzone_n'], subzone_2008['pln_area_n']))
# pairs_2014 = set(zip(subzone_2014['subzone_n'], subzone_2014['pln_area_n']))
# pairs_2019 = set(zip(subzone_2019['subzone_n'], subzone_2019['pln_area_n']))

# # 3. Find Differences
# only_in_2008 = pairs_2008 - pairs_2014
# only_in_2014 = pairs_2014 - pairs_2019
# only_in_2019 = pairs_2019 - pairs_2014

# # 4. Print results
# print(f"Count 2008: {len(pairs_2008)} | Count 2014: {len(pairs_2014)} | Count 2019: {len(pairs_2019)}")
# print("-" * 50)

# if only_in_2008:
#     print(f"Pairs only in 2008 ({len(pairs_2008)}):")
#     for s, p in sorted(pairs_2008):
#         print(f" - {s} ({p})")
# else:
#     print("No unique 2014 pairs found.")


# if only_in_2014:
#     print(f"Pairs only in 2014 ({len(only_in_2014)}):")
#     for s, p in sorted(only_in_2014):
#         print(f" - {s} ({p})")
# else:
#     print("No unique 2014 pairs found.")

# print("-" * 50)

# if only_in_2019:
#     print(f"Pairs only in 2019 ({len(only_in_2019)}):")
#     for s, p in sorted(only_in_2019):
#         print(f" - {s} ({p})")
# else:
#     print("No unique 2019 pairs found.")